In [1]:
import os
import glob
import pandas as pd
import numpy as np

SEMILLA = 42

# Buscar el CSV en los archivos de Colab
RUTA = glob.glob(
    "/content/**/plantvillage_tomate_metadata_auditado.csv",
    recursive=True
)

if not RUTA:
    raise FileNotFoundError("No se ha encontrado el archivo CSV en Colab.")

# Cargar el archivo encontrado
df = pd.read_csv(RUTA[0])

print("Archivo encontrado:", RUTA[0])
print("Dimensiones del conjunto:", df.shape)

df.info()

FileNotFoundError: No se ha encontrado el archivo CSV en Colab.

In [ ]:
print("Distribución de las particiones:")
print(df["particion_auditada"].value_counts())

print("\nDistribución de las clases:")
print(df["clase"].value_counts())

print("\nImágenes sin correspondencia en leaf-map:")
print((~df["existe_en_leaf_map"]).sum())

print("\nRegistros marcados como posible fuga:")
print(df["posible_fuga"].sum())

In [ ]:
# Separar las particiones originales
train_original = df[df["particion_auditada"] == "train"].copy()
test_original = df[df["particion_auditada"] == "test"].copy()

# Comprobar hojas identificadas que aparecen en ambas particiones
hojas_train = set(train_original.loc[
    train_original["existe_en_leaf_map"], "leaf_id"
])

hojas_test = set(test_original.loc[
    test_original["existe_en_leaf_map"], "leaf_id"
])

hojas_compartidas = hojas_train.intersection(hojas_test)

print("Hojas identificadas en train:", len(hojas_train))
print("Hojas identificadas en test:", len(hojas_test))
print("Hojas compartidas entre train y test:", len(hojas_compartidas))

# Revisar los registros marcados como posible fuga
print("\nPosibles fugas por partición:")
print(df.groupby("particion_auditada")["posible_fuga"].sum())

# Revisar las imágenes sin correspondencia en leaf-map
print("\nSin correspondencia en leaf-map por partición:")
print(
    df[~df["existe_en_leaf_map"]]
    .groupby("particion_auditada")
    .size()
)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Separar los datos originales
train_original = df[df["particion_auditada"] == "train"].copy()
test = df[df["particion_auditada"] == "test"].copy()

# Crear grupos: hoja identificada o imagen individual si no hay leaf-map
train_original["grupo"] = np.where(
    train_original["existe_en_leaf_map"],
    train_original["leaf_id"],
    train_original["image_path"]
)

# Dividir entrenamiento y validación por grupos
divisor = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEMILLA
)

indices_ent, indices_val = next(
    divisor.split(
        train_original,
        groups=train_original["grupo"]
    )
)

train = train_original.iloc[indices_ent].copy()
val = train_original.iloc[indices_val].copy()

# Mostrar tamaños
print("Entrenamiento:", len(train))
print("Validación:", len(val))
print("Test:", len(test))
print("Total:", len(train) + len(val) + len(test))

In [ ]:
# 1. Comprobar que no hay grupos compartidos
grupos_train = set(train["grupo"])
grupos_val = set(val["grupo"])

print("Grupos compartidos entre train y validación:",
      len(grupos_train.intersection(grupos_val)))

# 2. Comprobar las clases en cada conjunto
print("\nClases en entrenamiento:")
print(train["clase"].value_counts())

print("\nClases en validación:")
print(val["clase"].value_counts())

print("\nClases en test:")
print(test["clase"].value_counts())

# 3. Comprobar posibles fugas
print("\nPosibles fugas:")
print("Train:", train["posible_fuga"].sum())
print("Validación:", val["posible_fuga"].sum())
print("Test:", test["posible_fuga"].sum())

In [ ]:
# Mostrar los registros marcados como posible fuga
columnas_revision = [
    "image_path",
    "clase",
    "image_identifier",
    "leaf_id",
    "particion_auditada",
    "posible_fuga"
]

print(
    df[df["posible_fuga"]][columnas_revision]
    .to_string(index=False)
)

In [ ]:
# Revisar los grupos marcados como posible fuga
revision = pd.concat([
    train.assign(conjunto="entrenamiento"),
    val.assign(conjunto="validación")
])

revision_fugas = revision[revision["posible_fuga"]]

print(
    revision_fugas[
        ["leaf_id", "clase", "conjunto", "image_path"]
    ].sort_values("leaf_id").to_string(index=False)
)

print("\nNúmero de grupos por conjunto:")
print(
    revision_fugas.groupby("conjunto")["leaf_id"].nunique()
)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Recuperar el conjunto original de entrenamiento
train_original = df[df["particion_auditada"] == "train"].copy()
test = df[df["particion_auditada"] == "test"].copy()

# Crear identificador de grupo
train_original["grupo"] = train_original["leaf_id"]

# Dividir por grupos
divisor = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEMILLA
)

indices_ent, indices_val = next(
    divisor.split(
        train_original,
        groups=train_original["grupo"]
    )
)

train = train_original.iloc[indices_ent].copy()
val = train_original.iloc[indices_val].copy()

# Comprobar grupos compartidos
grupos_compartidos = (
    set(train["grupo"]) & set(val["grupo"])
)

print("Entrenamiento:", len(train))
print("Validación:", len(val))
print("Test:", len(test))
print("Grupos compartidos:", len(grupos_compartidos))

In [ ]:
# Comprobar que las parejas fallback están en un único conjunto
revision = pd.concat([
    train.assign(conjunto="entrenamiento"),
    val.assign(conjunto="validación")
])

fugas = revision[revision["posible_fuga"]]

print("Grupos fallback por conjunto:")
print(
    fugas.groupby("leaf_id")["conjunto"].nunique()
)

print("\nGrupos compartidos entre train y validación:")
print(
    len(set(train["leaf_id"]) & set(val["leaf_id"]))
)

print("\nPosibles fugas en cada conjunto:")
print("Entrenamiento:", train["posible_fuga"].sum())
print("Validación:", val["posible_fuga"].sum())
print("Test:", test["posible_fuga"].sum())

**2. Construir el baseline**

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report

# Crear el baseline
baseline = DummyClassifier(strategy="most_frequent")

# Entrenar solamente con entrenamiento
baseline.fit(train["clase"], train["clase"])

# Evaluar con validación
predicciones = baseline.predict(val["clase"])

# Calcular métricas
f1_macro = f1_score(
    val["clase"],
    predicciones,
    average="macro",
    zero_division=0
)

accuracy = accuracy_score(
    val["clase"],
    predicciones
)

print("Clase que predice el baseline:", baseline.classes_[0])
print("F1-macro:", round(f1_macro, 4))
print("Accuracy:", round(accuracy, 4))

print("\nInforme de clasificación:")
print(classification_report(
    val["clase"],
    predicciones,
    zero_division=0
))

**F1-macro**
0,0442

**Accuracy**
0,2834

**Interpretación:** el baseline acierta el 28,34 % de las imágenes porque siempre predice la clase mayoritaria. El F1-macro es muy bajo porque no reconoce las otras nueve clases.

In [ ]:
clase_mayoritaria = train["clase"].value_counts().idxmax()

print("Clase mayoritaria:", clase_mayoritaria)
print("F1-macro:", round(f1_macro, 4))
print("Accuracy:", round(accuracy, 4))

### Conclusión del baseline

Se ha construido un modelo de referencia DummyClassifier con
estrategia `most_frequent`, que predice siempre la clase mayoritaria.

El modelo obtiene un F1-macro de 0,0442 y un accuracy de 0,2834
sobre el conjunto de validación.

Estos resultados muestran que predecir siempre una única clase
no es suficiente para clasificar correctamente las diez categorías
de hojas de tomate.

Este baseline servirá como referencia para comparar los modelos
de clasificación de imágenes que construiremos posteriormente.

In [ ]:
import os
import json
import zipfile
from google.colab import files

# Crear carpeta de resultados
CARPETA = "resultados_baseline"
os.makedirs(CARPETA, exist_ok=True)

# Añadir identificador de grupo al test
test_guardar = test.copy()
test_guardar["grupo"] = test_guardar["leaf_id"]

# Añadir nombre del conjunto
train_guardar = train.copy()
train_guardar["conjunto"] = "entrenamiento"

val_guardar = val.copy()
val_guardar["conjunto"] = "validacion"

test_guardar["conjunto"] = "test"

# Guardar los tres conjuntos
train_guardar.to_csv(
    f"{CARPETA}/entrenamiento.csv", index=False
)

val_guardar.to_csv(
    f"{CARPETA}/validacion.csv", index=False
)

test_guardar.to_csv(
    f"{CARPETA}/test.csv", index=False
)

# Guardar resultados del baseline
resultados = {
    "proyecto": "AgroVision IA",
    "equipo": "Las Guerreras",
    "modelo": "DummyClassifier",
    "estrategia": "most_frequent",
    "clase_mayoritaria": train["clase"].value_counts().idxmax(),
    "f1_macro": float(f1_macro),
    "accuracy": float(accuracy),
    "evaluacion": "validacion",
    "semilla": SEMILLA
}

with open(f"{CARPETA}/resultados_baseline.json", "w",
          encoding="utf-8") as archivo:
    json.dump(resultados, archivo, ensure_ascii=False, indent=4)

# Guardar resumen de la partición
resumen = {
    "entrenamiento": len(train),
    "validacion": len(val),
    "test": len(test),
    "total": len(train) + len(val) + len(test),
    "grupos_compartidos_train_validacion": len(
        set(train["grupo"]) & set(val["grupo"])
    ),
    "posibles_fugas_train": int(train["posible_fuga"].sum()),
    "posibles_fugas_validacion": int(val["posible_fuga"].sum()),
    "posibles_fugas_test": int(test["posible_fuga"].sum()),
    "imagenes_sin_leaf_map": int((~df["existe_en_leaf_map"]).sum())
}

with open(f"{CARPETA}/resumen_particion.json", "w",
          encoding="utf-8") as archivo:
    json.dump(resumen, archivo, ensure_ascii=False, indent=4)

# Crear ZIP
with zipfile.ZipFile(
    "AgroVision_IA_datos_baseline.zip", "w",
    zipfile.ZIP_DEFLATED
) as zipf:
    for archivo in os.listdir(CARPETA):
        zipf.write(
            f"{CARPETA}/{archivo}",
            arcname=archivo
        )

print("Archivos guardados correctamente:")
for archivo in os.listdir(CARPETA):
    print("-", archivo)

print("\nResumen:", resumen)

# Descargar el ZIP al ordenador
files.download("AgroVision_IA_datos_baseline.zip")